# Phase 1 — MedQA Uncertainty Experiment

Two-turn active-clarification pipeline on the MedQA held-out set.

**Pipeline per record:**
1. Model sees `ehr_summary` + `question` (no answer options) → generates clarifying question + preliminary assessment + confidence
2. Patient simulator answers the CQ from the combined `patient_context` / `nurse_context` / `specialist_context`
3. Model sees presentation + question + clarifying exchange + answer options → updated assessment + confidence

**Key outputs:** preliminary/updated correctness, confidence delta, clarifying question (classified by judge in Phase 2)

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

# ── Dataset config ────────────────────────────────────────────────────────
DATASET          = "medqa"
ROOT             = Path("../../").resolve()
PROMPTS_DIR      = ROOT / "prompts"  / DATASET
DATASETS_DIR     = ROOT / "datasets" / DATASET
OUTPUTS_DIR      = ROOT / "outputs"  / DATASET

CASES_PATH       = DATASETS_DIR / "unseen_100.jsonl"
INSTRUCTION_FILE = PROMPTS_DIR  / "phase1_instruction.txt"
OUTPUT_CSV       = OUTPUTS_DIR  / "phase1_results.csv"

# ── Run config ────────────────────────────────────────────────────────────
N_RECORDS        = 100
MODEL_ID         = "gemini-2.5-flash"
REQUEST_INTERVAL = 1.0     # 1 s between calls — same as CLAMBER eval, no rate issues
RANDOM_SEED      = 42
# ─────────────────────────────────────────────────────────────────────────

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"ROOT:        {ROOT}")
print(f"Cases:       {CASES_PATH}")
print(f"Instruction: {INSTRUCTION_FILE}")
print(f"Output CSV:  {OUTPUT_CSV}")

ROOT:        D:\final_project\pilot_study
Cases:       D:\final_project\pilot_study\datasets\medqa\unseen_100.jsonl
Instruction: D:\final_project\pilot_study\prompts\medqa\phase1_instruction.txt
Output CSV:  D:\final_project\pilot_study\outputs\medqa\phase1_results.csv


In [2]:
import importlib
import json
import logging
import os

import pandas as pd
from IPython.display import display, Markdown

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv, parse_json_response, format_answer_choices
from src.providers import GeminiProvider
from src.pipeline import Phase1Pipeline, PatientSimulator, TURN_0_SCHEMA, TURN_1_SCHEMA
from src.utils import SafetyBlockError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded. src modules reloaded.")

Environment loaded. src modules reloaded.


## Smoke Test

In [3]:
provider_smoke = GeminiProvider(model_id=MODEL_ID)
response = provider_smoke.call(
    system_instruction="You are a test assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0,
    max_tokens=512,   # thinking model needs headroom for reasoning tokens
)
print(f"Smoke test response: {response.strip()}")
assert len(response.strip()) > 0, "Smoke test failed — empty response"
print("Smoke test passed.")

17:03:41 [INFO] src.providers — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta


17:03:41 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


17:03:42 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Smoke test response: SMOKE TEST PASSED
Smoke test passed.


## Load Dataset

In [4]:
with open(CASES_PATH, encoding="utf-8") as f:
    raw_cases = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_cases)} cases from {CASES_PATH.name}")

# Build records in pipeline format.
# Simulator context = patient + nurse + specialist (excludes full_patient_context
# which contains an explicit teaching point that reveals the answer).
records = []
for c in raw_cases:
    simulator_context = "\n\n---\n\n".join(
        ctx for ctx in [
            c.get("patient_context", ""),
            c.get("nurse_context", ""),
            c.get("specialist_context", ""),
        ] if ctx and ctx.strip()
    )
    records.append({
        "id":               c["case_id"],
        "ehr_summary":      c["ehr_summary"],
        "question":         c["question"],
        "options":          c["options"],
        "correct_option":   c["correct_option"],
        "correct_answer":   c["correct_answer"],
        "simulator_context": simulator_context,
        "difficulty":       c.get("difficulty", ""),
    })

print(f"Records prepared: {len(records)}")
print(f"\nDifficulty distribution:")
diff_counts = pd.Series([r['difficulty'] for r in records]).value_counts()
print(diff_counts.to_string())

print("\n--- Sample record ---")
r = records[0]
print(f"ID:            {r['id']} | difficulty: {r['difficulty']}")
print(f"EHR summary:   {r['ehr_summary']}")
print(f"Question:      {r['question']}")
print(f"Options:       {r['options']}")
print(f"Correct:       {r['correct_option']} | {r['correct_answer']}")
print(f"Simulator ctx: {r['simulator_context'][:200]}...")

Loaded 100 cases from unseen_100.jsonl
Records prepared: 100

Difficulty distribution:
easy    100

--- Sample record ---
ID:            medqa_1149 | difficulty: easy
EHR summary:   21 months-year-old male presenting with: leg deformities since starting to walk
Question:      Which of the following is the most appropriate next step in management?
Options:       {'A': 'Vitamin D supplementation', 'B': 'Reassurance and follow-up', 'C': 'X-ray of the lower extremities', 'D': 'Bracing of the lower extremities'}
Correct:       B | Reassurance and follow-up
Simulator ctx: **Simulated Patient Profile:**
---

**Demographics and Chief Complaint:**
- **Name:** Jacob M.
- **Age:** 21 months
- **Sex:** Male
- **Race/Ethnicity:** Caucasian
- **Accompanied by:** Mother
- **Chi...


## Preview Instruction

In [5]:
print(INSTRUCTION_FILE.read_text(encoding="utf-8"))

You are an experienced clinician seeing a new patient. You have been given a brief patient presentation, a clinical question, and the answer options.

Your task has three parts. Complete all three and return them as a valid JSON object.

Part 1 — Clarifying Question:
Based on the information provided, ask exactly one clarifying question that would most help you choose between the answer options. This should be the question that — if answered — would most change or sharpen your thinking about which option is correct. It can target any aspect of the case: a symptom detail, a timeline, a patient preference, a specific finding, or an ambiguity in the presentation. Ask it as you would in a real clinical encounter — naturally and concisely.

Part 2 — Preliminary Answer:
Select your best answer from the options provided. Return exactly one letter: A, B, C, or D. Commit to a best guess even with limited data.

Part 3 — Confidence:
Rate your confidence in your preliminary answer from 0 (complet

## Dry Run — Single Record
Verify the full two-turn flow on one record before running everything.

In [6]:
provider  = GeminiProvider(model_id=MODEL_ID)
simulator = PatientSimulator(provider)
instruction = INSTRUCTION_FILE.read_text(encoding="utf-8").strip()

test = records[1]   # pick second record for variety
print(f"Testing on: {test['id']} | difficulty={test['difficulty']}")
print(f"EHR:  {test['ehr_summary']}")
print(f"Q:    {test['question']}")
print(f"Options: {test['options']}")
print()

# Turn 0 — include options so CQ targets the answer choices
user_msg_0 = (
    f"Patient presentation:\n{test['ehr_summary'].strip()}\n\n"
    f"Clinical question:\n{test['question'].strip()}\n\n"
    f"Answer options:\n{format_answer_choices(test['options'])}"
)
raw_0 = provider.call(
    system_instruction=instruction,
    user_message=user_msg_0,
    temperature=0.0,
    max_tokens=4096,
    expect_json=TURN_0_SCHEMA,
)
parsed_0 = parse_json_response(raw_0)
print("=== TURN 0 ===")
print(json.dumps(parsed_0, indent=2))

# Patient simulation
cq = parsed_0["clarifying_question"]
patient_answer = simulator.answer(cq, test["simulator_context"])
print(f"\n=== PATIENT RESPONSE ===\n{patient_answer}")

# Turn 1
from src.pipeline import _POST_CLARIFICATION_INSTRUCTION
user_msg_1 = (
    f"Patient presentation:\n{test['ehr_summary'].strip()}\n\n"
    f"Your clarifying question:\n{cq}\n\n"
    f"Patient's answer:\n{patient_answer}\n\n"
    f"Clinical question:\n{test['question'].strip()}\n\n"
    f"Answer choices:\n{format_answer_choices(test['options'])}"
)
raw_1 = provider.call(
    system_instruction=_POST_CLARIFICATION_INSTRUCTION,
    user_message=user_msg_1,
    temperature=0.0,
    max_tokens=4096,
    expect_json=TURN_1_SCHEMA,
)
parsed_1 = parse_json_response(raw_1)
print(f"\n=== TURN 1 ===\n{json.dumps(parsed_1, indent=2)}")
print(f"\nCorrect answer: {test['correct_option']} | {test['correct_answer']}")

17:03:42 [INFO] src.providers — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta


17:03:42 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


Testing on: medqa_0638 | difficulty=easy
EHR:  68-year-old female presenting with: abdominal pain and watery, foul-smelling diarrhea
Q:    Sterilization with which of the following agents is most likely to prevent transmission of this pathogen to the next patient who will occupy her room?
Options: {'A': 'Chlorine-based solution', 'B': 'Iodine-based solution', 'C': 'Isopropanol-based solution', 'D': 'Quaternary amine-based solution'}



17:03:51 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


17:03:51 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


...

17:03:52 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


17:03:52 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.



=== PATIENT RESPONSE ===
Yes, stool studies showed growth of Clostridioides difficile and tested positive for C. difficile toxin A/B, confirming it as the cause of the diarrhea.


17:03:54 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



=== TURN 1 ===
{
  "updated_assessment": "A",
  "updated_confidence": 95
}

Correct answer: A | Chlorine-based solution


## Run Full Experiment
Processes all 100 held-out cases. Resumes automatically if interrupted.

In [7]:
provider = GeminiProvider(model_id=MODEL_ID)

pipeline = Phase1Pipeline(
    provider=provider,
    instruction_file=INSTRUCTION_FILE,
    output_csv=OUTPUT_CSV,
    request_interval=REQUEST_INTERVAL,
)

pipeline.run(records)

17:03:54 [INFO] src.providers — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta


17:03:54 [INFO] src.pipeline — Phase1Pipeline ready — provider=gemini model=gemini-2.5-flash output=D:\final_project\pilot_study\outputs\medqa\phase1_results.csv


17:03:54 [INFO] src.pipeline — [1/100] Processing medqa_1149


17:03:54 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


17:04:15 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


...

17:40:14 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


17:40:16 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


17:40:16 [INFO] src.pipeline —   Updated: A (conf=95)


17:40:17 [INFO] src.pipeline — Phase 1 complete — total=100 succeeded=100 skipped=0 failed=0


17:40:17 [INFO] src.pipeline — Results saved to: D:\final_project\pilot_study\outputs\medqa\phase1_results.csv


## Results Summary

In [8]:
results_df = pd.read_csv(OUTPUT_CSV)
valid = results_df[~results_df["was_blocked"]]

print(f"Total records:           {len(results_df)}")
print(f"Blocked:                 {results_df['was_blocked'].sum()}")
print(f"Valid:                   {len(valid)}")
print()
print(f"Preliminary correct:     {valid['is_correct_preliminary'].sum()} / {len(valid)} "
      f"({valid['is_correct_preliminary'].mean():.1%})")
print(f"Updated correct:         {valid['is_correct_updated'].sum()} / {len(valid)} "
      f"({valid['is_correct_updated'].mean():.1%})")
print()
print(f"Mean prelim confidence:  {valid['preliminary_confidence'].mean():.1f}")
print(f"Mean updated confidence: {valid['updated_confidence'].mean():.1f}")
print(f"Mean confidence delta:   {valid['confidence_delta'].mean():.1f}")
print()

print("Confidence delta distribution:")
print(f"  Increased: {(valid['confidence_delta'] > 0).sum()}")
print(f"  No change: {(valid['confidence_delta'] == 0).sum()}")
print(f"  Decreased: {(valid['confidence_delta'] < 0).sum()}")
print()

print("Per-difficulty breakdown:")
display(valid.groupby('difficulty')[['is_correct_preliminary','is_correct_updated','confidence_delta']]
        .agg({'is_correct_preliminary':'mean','is_correct_updated':'mean','confidence_delta':'mean'})
        .round(3))

Total records:           100
Blocked:                 0
Valid:                   100

Preliminary correct:     76 / 100 (76.0%)
Updated correct:         82 / 100 (82.0%)

Mean prelim confidence:  72.7
Mean updated confidence: 91.7
Mean confidence delta:   19.0

Confidence delta distribution:
  Increased: 86
  No change: 9
  Decreased: 5

Per-difficulty breakdown:


,is_correct_preliminary,is_correct_updated,confidence_delta
difficulty,,,
easy,0.76,0.82,18.97


In [9]:
# Full per-record detail for qualitative inspection
display(Markdown("**Per-record results (first 10):**"))
for _, row in results_df.head(10).iterrows():
    print("="*80)
    print(f"ID:          {row['id']} | difficulty={row['difficulty']}")
    print(f"EHR:         {row['ehr_summary']}")
    print(f"CQ:          {row['clarifying_question']}")
    print(f"Patient:     {row['patient_response']}")
    print(f"Prelim:      {row['preliminary_assessment']} (conf={row['preliminary_confidence']})")
    print(f"Updated:     {row['updated_assessment']} (conf={row['updated_confidence']})")
    print(f"Correct:     {row['correct_option']} | {row['correct_answer']}")
    print(f"Prelim OK:   {row['is_correct_preliminary']}  |  Updated OK: {row['is_correct_updated']}  |  Δconf: {row['confidence_delta']:+}")
    print()

**Per-record results (first 10):**

ID:          medqa_1149 | difficulty=easy
EHR:         21 months-year-old male presenting with: leg deformities since starting to walk
CQ:          Is the bowing symmetrical, or is one leg more affected than the other?
Patient:     The bowing of Jacob's legs is symmetric, with both tibiae mildly affected.
Prelim:      C (conf=65)
Updated:     B (conf=90)
Correct:     B | Reassurance and follow-up
Prelim OK:   False  |  Updated OK: True  |  Δconf: +25

ID:          medqa_0638 | difficulty=easy
EHR:         68-year-old female presenting with: abdominal pain and watery, foul-smelling diarrhea
CQ:          Is the diarrhea confirmed to be caused by Clostridioides difficile?
Patient:     Yes, stool studies showed growth of Clostridioides difficile and tested positive for C. difficile toxin A/B, confirming it as the cause of the diarrhea.
Prelim:      A (conf=90)
Updated:     A (conf=95)
Correct:     A | Chlorine-based solution
Prelim OK:   True  |  Updated OK: True  |  Δconf: +5

ID:        